In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

train_dataset = datasets.SVHN(
    root='./data',
    split='train',
    download=True,
    transform=transform
)

test_dataset = datasets.SVHN(
    root='./data',
    split='test',
    download=True,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=64, shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=64, shuffle=False
)

100%|██████████| 182M/182M [00:12<00:00, 14.7MB/s]
100%|██████████| 64.3M/64.3M [00:02<00:00, 22.5MB/s]


In [ ]:
model = nn.Sequential(
    nn.Flatten(),

    nn.Linear(3 * 32 * 32, 128),
    nn.ReLU(),

    nn.Linear(128, 64),
    nn.ReLU(),

    nn.Linear(64, 10)  # No Softmax!
)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 10
losses = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        # Fix labels (10 → 0)
        labels = labels % 10

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    losses.append(epoch_loss)

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")

Epoch [1/10], Loss: 1.2055
Epoch [2/10], Loss: 0.8312
Epoch [3/10], Loss: 0.7161
Epoch [4/10], Loss: 0.6538
Epoch [5/10], Loss: 0.6135
Epoch [6/10], Loss: 0.5813
Epoch [7/10], Loss: 0.5553
Epoch [8/10], Loss: 0.5348
Epoch [9/10], Loss: 0.5193
Epoch [10/10], Loss: 0.5042


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        labels = labels % 10

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"\nTest Accuracy: {accuracy:.2f}%")


Test Accuracy: 78.53%
